# Descriptive Statistics, Correlation, t-Test & ANOVA

1. Descriptive Statistics
2. Crosstabulation
3. Pearson Correlation
4. Two-Sample t-Test
5. One-Way ANOVA
6. Effect Sizes
7. Bivariate Regression

In [ ]:
# Imports
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr, ttest_ind, levene
from scipy.stats import f as f_dist

pd.set_option("display.max_columns", None)

In [ ]:
# Load Dataset
df = pd.read_csv("")

## 1. Descriptive Statistics

Calculate the mean, standard deviation, variance, minimum, maximum, and standard error of the mean for the numerical variables.

In [ ]:
# Select numerical columns
num = df.select_dtypes(include="number")

# Create a DataFrame containing descriptive statistics for all numerical variables
descriptive = pd.DataFrame({
    "N": num.count(),  # Count the non-missing values
    "Mean": num.mean(),  # Calculate the arithmetic mean
    "Std. Deviation": num.std(),  # Calculate the standard deviation
    "Variance": num.var(),  # Calculate the variance
    "Minimum": num.min(),  # Identify the minimum value
    "Maximum": num.max(),  # Identify the maximum value
    "S.E. Mean": num.sem(),  # Calculate the standard error of the mean
})

# Output
print("\nDescriptive Statistics")
print("-----------------------------")
print(descriptive.round(3))

## 2. Crosstab

Create a crosstab with Education Level as rows and Marital Status as columns. How many observations are by singles with highschool degree?

In [ ]:
# Create a crosstab with Education Level and Marital Status
crosstab = pd.crosstab(
    df["Education_Level"],
    df["Marital_Status"]
)

#  Output
print("\nCrosstab: Education Level x Marital Status")
print("-----------------------------")
print(crosstab)

## 3. Correlation

Compute the Pearson correlation between Income and Average Spend. How do you interpret the result?

In [ ]:
# Compute pearson r between Income and Average Spend
r, p = pearsonr(
    df["Income"],
    df["Average_Spend"]
)

# Output
print("\nCorrelation: Income x Average Spend")
print("-----------------------------")
print(f"r = {r:.3f}")
print(f"p = {p:.3f}")

In [ ]:
# Scatterplot
# Create a figure with a width of 8 inches and a height of 5 inches
plt.figure(figsize=(8, 5))

# Create a scatter plot with a regression line
sns.regplot(
    data=df, # Use the DataFrame as the data source
    x="Income", # Set Income as the x-axis variable
    y="Average_Spend", # Set Average_Spend as the y-axis variable
    scatter_kws={"alpha": 0.5}, # Set the transparency of the data points
    line_kws={"color": "steelblue"} # Set the regression line color
)

# Add a title to the plot
plt.title("Income vs. Average Spend")

# Label the x-axis
plt.xlabel("Income")

# Label the y-axis
plt.ylabel("Average Spend")

# Adjust the layout to prevent overlapping elements
plt.tight_layout()

# Output
plt.show()

## 4. Two-Sample t-Test

The t-statistic expresses the group difference relative to its uncertainty (**signal-to-noise ratio**):

$$
t = \frac{\text{MeanDiff}}{SE} \quad \rightarrow \quad \frac{\text{Signal}}{\text{Noise}}
$$

with the standard error

$$
SE = \frac{SD}{\sqrt{N}}
$$

Levene's test (equal variances test):

- $p > 0.05$ → variances are assumed to be equal
- $p \leq 0.05$ → variances are assumed not to be equal

**Test whether men and women differ in their average spending.**

Note: For this test, the subjects should be randomly assigned to two groups, so that any difference in response is due to the treatment (or lack of treatment) and not to other factors (as in our example here).

In [ ]:
# Select Average_Spend values for female customers and remove missing values
female = df.loc[df["Gender"] == "Female", "Average_Spend"].dropna()

# Select Average_Spend values for male customers and remove missing values
male = df.loc[df["Gender"] == "Male", "Average_Spend"].dropna()

# Levene's Test for Equality of Variances
lev_stat, lev_p = levene(male, female, center="mean")

# t-Tests for Equality of Means
# Perform an independent t-test assuming equal variances (Student's t-test)
equal = ttest_ind(male, female, equal_var=True)
# Perform an independent t-test without assuming equal variances (Welch's t-test)
welch = ttest_ind(male, female, equal_var=False)

# Mean Difference
diff = male.mean() - female.mean()

# Confidence Intervals
# Calculate the 95% confidence interval for the equal-variance t-test
ci_equal = equal.confidence_interval(confidence_level=0.95)
# Calculate the 95% confidence interval for Welch's t-test
ci_welch = welch.confidence_interval(confidence_level=0.95)

# Output
print("Levene's Test for Equality of Variances")
print("---------------------------------------")
print(f"Statistic: {lev_stat:.3f}")
print(f"p-value:   {lev_p:.3f}")

print("\nGroup Statistics")
print("---------------------------------------")
print(f"n men / women: {len(male)} / {len(female)}")
print(f"Mean men:      {male.mean():.2f}")
print(f"Mean women:    {female.mean():.2f}")
print(f"Difference:    {diff:.2f}")

print("\nt-Test for Equality of Means")
print("---------------------------------------")

print("Equal variances assumed")
print(f"t = {equal.statistic:.3f}")
print(f"df = {equal.df:.0f}")
print(f"p = {equal.pvalue:.3f}")
print(f"95% CI: [{ci_equal.low:.2f}, {ci_equal.high:.2f}]")

print("\nEqual variances not assumed (Welch)")
print(f"t = {welch.statistic:.3f}")
print(f"df = {welch.df:.2f}")
print(f"p = {welch.pvalue:.3f}")
print(f"95% CI: [{ci_welch.low:.2f}, {ci_welch.high:.2f}]")

## 5. One-Way ANOVA

Perform a one-way ANOVA to examine whether Average Spend differs between men and women.

Note: For this test, the subjects should be randomly assigned to two groups, so that any difference in response is due to the treatment (or lack of treatment) and not to other factors (as in our example here).

In [ ]:
# Store the two groups to be compared
groups = [female, male]

# Count the number of groups
k = len(groups)

# Calculate group sizes, total sample size and overall mean
n_groups = [g.count() for g in groups]
N = sum(n_groups)
grand_mean = pd.concat(groups).mean()

# Sums of squares
# Calculate the between-group sum of squares
SS_between = sum(n * (g.mean() - grand_mean) ** 2 for n, g in zip(n_groups, groups))
# Calculate the within-group sum of squares
SS_within = sum(((g - g.mean()) ** 2).sum() for g in groups)

# Degrees of freedom
# Calculate the degrees of freedom between groups
df_between = k - 1
# Calculate the degrees of freedom within groups
df_within = N - k

# Mean squares
# Calculate the mean squares between groups
MS_between = SS_between / df_between
# Calculate the mean squares within groups
MS_within = SS_within / df_within

# Calculate the F-statistic and the corresponding p-value
F = MS_between / MS_within
p_value = f_dist.sf(F, df_between, df_within)

# Assumption: homogeneity of variances

# Output
print("\nOne-Way ANOVA")
print("-----------------------------")
print(f"SS Between:  {SS_between:.3f}")
print(f"SS Within:   {SS_within:.3f}")
print(f"df Between:  {df_between}")
print(f"df Within:   {df_within}")
print(f"MS Between:  {MS_between:.3f}")
print(f"MS Within:   {MS_within:.3f}")
print(f"F:           {F:.3f}")
print(f"p = {p_value:.3f}")

**Sum of squares between groups**:
- Variation in Average_Spend explained by factor (here Gender)

**Sum of squares within groups**:
- Variation in Average_Spend that cannot be explained by factor

## 6. Effect Sizes

Calculate Eta-squared (η²) and Omega-squared (ω²).

**Eta-squared**: Share of variation in Average_Spend explained by factor (here Gender)
The value of eta-squared ranges from 0 to 1:
$$
0 \leq \eta^2 \leq 1
$$

Rules of thumb for the effect size of eta-squared:
- 0.01: Small effect
- 0.06: Medium effect
- 0.14 or higher: Large effect

**Omega-squared**:
- Between - and + 1, 0 indicates no effect
- Formula takes sample size into account, hence more robust to small samples

In [ ]:
# Calculate the total sum of squares
SS_total = SS_between + SS_within

# Calculate the Eta-squared
eta_squared = SS_between / SS_total

# Calculate the Omega-squared
omega_squared = (SS_between - df_between * MS_within) / (SS_total + MS_within)

# Output
print("\nEffect Sizes")
print("-----------------------------")
print(f"SS Total:            {SS_total:.3f}")
print(f"Eta-squared (η²):    {eta_squared:.3f}")
print(f"Omega-squared (ω²):  {omega_squared:.3f}")

## 7. Bivariate Regression

Estimate a bivariate linear regression with Gender as a dummy-coded independent variable and Average Spend as the dependent variable.

The dummy variable is coded as:

- Female = 0
- Male = 1

In [ ]:
# Dummy variable: Female = 0, Male = 1
data = df.assign(Male=(df["Gender"] == "Male").astype(int))

In [ ]:
# Add a constant term to the independent variable to estimate the intercept
X = sm.add_constant(data["Male"])

# Define the dependent variable
y = data["Average_Spend"]

# Fit an Ordinary Least Squares (OLS) regression model
model = sm.OLS(y, X).fit()

# Output
print("\nBivariate Regression")
print("-----------------------------")
print(model.summary())

**R Square**: Share of variation in Average_Spend explained by model (here only Gender-dummy)